# Lab 4: Generating CDFs

We pick four simulations from the lab, run each one, plot the ECDF of the simulated values, and compare it against the theoretical `scipy.stats` distribution it converges to.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import uniform, norm, expon

## Q1: Sampling from a Grid

- Draw points uniformly at random from an equally spaced grid on $[0,1]$.
- As the number of grid points $K$ grows, the grid fills in $[0,1]$, so the draws should approach a continuous $\text{Uniform}[0,1]$.

In [ ]:
rng = np.random.default_rng(seed=100)

n = 5_000
x_grid = np.linspace(0, 1, 200)

for K in [4, 50]:
    grid = np.arange(1, K + 1) / K
    draws = rng.choice(grid, size=n)

    plt.figure(figsize=(6, 4))
    sns.ecdfplot(x=draws, label=f"Simulated ECDF (K={K})")
    plt.plot(x_grid, uniform.cdf(x_grid), label="Uniform(0,1) CDF")
    plt.title(f"Grid sampling, K={K}")
    plt.legend()
    plt.show()

**What distribution does this approach?** As $K$ increases, the grid points get closer together and the ECDF converges to the $\text{Uniform}[0,1]$ CDF.

## Q2: Distribution of the Sample Mean

- Draw $n=32$ values from $\text{Uniform}[-\sqrt3, \sqrt3]$ (mean 0, variance 1).
- Compute the sample mean and scale it by $\sqrt{n}$.
- Repeat $b=5{,}000$ times and look at the ECDF of the scaled means.

In [ ]:
rng = np.random.default_rng(seed=100)

n = 32
b = 5_000
a = np.sqrt(3)

X = rng.uniform(-a, a, size=(b, n))
scaled_means = X.mean(axis=1) * np.sqrt(n)

x = np.linspace(-4, 4, 200)

plt.figure(figsize=(6, 4))
sns.ecdfplot(x=scaled_means, label="Simulated ECDF")
plt.plot(x, norm.cdf(x), label="Standard Normal CDF")
plt.title("Scaled sample mean, n=32")
plt.legend()
plt.show()

**What distribution does this approach?** By the Central Limit Theorem, $\sqrt{n}\,\bar X_n$ converges to a $\text{Normal}(0,1)$ distribution as $n$ increases, regardless of the underlying uniform distribution of the data.

## Q3: Standardized Count of Survivors

- Flip $n=40$ coins with probability of heads $p$.
- Standardize the count of heads: subtract the mean $np$ and divide by the standard deviation $\sqrt{np(1-p)}$.
- Repeat $b=5{,}000$ times and look at the ECDF.

In [ ]:
rng = np.random.default_rng(seed=100)

n = 40
p = 0.5
b = 5_000

heads = rng.binomial(n, p, size=b)
z = (heads - n * p) / np.sqrt(n * p * (1 - p))

x = np.linspace(-4, 4, 200)

plt.figure(figsize=(6, 4))
sns.ecdfplot(x=z, label="Simulated ECDF")
plt.plot(x, norm.cdf(x), label="Standard Normal CDF")
plt.title(f"Standardized binomial count, p={p}")
plt.legend()
plt.show()

In [ ]:
# How does the result change as p moves away from 0.5?
for p in [0.5, 0.05]:
    heads = rng.binomial(n, p, size=b)
    z = (heads - n * p) / np.sqrt(n * p * (1 - p))

    plt.figure(figsize=(6, 4))
    sns.ecdfplot(x=z, label=f"Simulated ECDF (p={p})")
    plt.plot(x, norm.cdf(x), label="Standard Normal CDF")
    plt.title(f"Standardized binomial count, p={p}")
    plt.legend()
    plt.show()

**What distribution does this approach?** As $n$ increases, the standardized count converges to a $\text{Normal}(0,1)$ distribution (another instance of the Central Limit Theorem, this time applied to a sum of Bernoulli trials).

**How do results change with $p$?** For $p=0.5$ the approximation is already very close to normal at $n=40$, since the underlying Bernoulli distribution is symmetric. For extreme $p$ (e.g. $p=0.05$), the Bernoulli distribution is highly skewed, so the standardized ECDF still shows visible skew/discreteness relative to the normal curve at this sample size — convergence to normality is slower.

## Q4: Discrete-Time Survival Process

- A process running in discrete time terminates in each interval of length `dt` with probability `r * dt`.
- Simulate the period of termination (a geometric random variable) and convert it to time by multiplying by `dt`.
- Repeat $n=5{,}000$ times and look at the ECDF as `dt` gets small.

In [ ]:
rng = np.random.default_rng(seed=100)

r = 1.0
dt = 1e-3
n = 5_000

p_term = r * dt
periods = rng.geometric(p_term, size=n)   # period in which termination occurs
time = periods * dt                       # convert to continuous time

x = np.linspace(0, 8, 200)

plt.figure(figsize=(6, 4))
sns.ecdfplot(x=time, label="Simulated ECDF")
plt.plot(x, expon.cdf(x, scale=1 / r), label="Exponential CDF")
plt.title(f"Discrete-time survival process, dt={dt}")
plt.legend()
plt.show()

In [ ]:
# How does the result change as r changes?
for r in [1.0, 3.0]:
    p_term = r * dt
    periods = rng.geometric(p_term, size=n)
    time = periods * dt

    plt.figure(figsize=(6, 4))
    sns.ecdfplot(x=time, label=f"Simulated ECDF (r={r})")
    plt.plot(x, expon.cdf(x, scale=1 / r), label=f"Exponential CDF (r={r})")
    plt.title(f"Discrete-time survival process, r={r}")
    plt.legend()
    plt.show()

**What distribution does this approach?** As `dt` goes to zero, the discrete-time termination process converges to a continuous $\text{Exponential}(r)$ distribution — this is the discrete-time analogue of the memoryless waiting-time process.

**How do results change with $r$?** Larger $r$ means a higher termination probability per interval, so termination happens sooner on average; the exponential distribution becomes more concentrated near 0 (mean $1/r$ shrinks as $r$ grows).